# 01 - Data Exploration: Blue Zones Historical Panel

This notebook explores the historical dataset covering 93 countries from 1960-2023.
All data is sourced from the World Bank API and WHO Global Health Observatory.

**Key questions:**
- What does our dataset look like?
- How complete is the coverage across countries and years?
- What are the basic distributions of key indicators?

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

PROJECT_DIR = os.path.dirname(os.path.abspath(os.getcwd()))
if not os.path.exists(os.path.join(PROJECT_DIR, 'data')):
    PROJECT_DIR = os.getcwd()
    if not os.path.exists(os.path.join(PROJECT_DIR, 'data')):
        PROJECT_DIR = os.path.dirname(PROJECT_DIR)

print(f'Project directory: {PROJECT_DIR}')

Project directory: /home/yeblad/Blue-Zones-Longevity-Analysis


## 1. Load the Historical Panel Data

In [2]:
df = pd.read_csv(os.path.join(PROJECT_DIR, 'data', 'historical', 'merged_historical_panel.csv'))
print(f'Dataset shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'Countries: {df["iso_code"].nunique()}')
print(f'Year range: {df["year"].min()} - {df["year"].max()}')
print(f'Blue Zone countries: {df[df["is_blue_zone"]==1]["iso_code"].nunique()}')
print()
df.head()

Dataset shape: 5,952 rows x 21 columns
Countries: 93
Year range: 1960 - 2023
Blue Zone countries: 5



,iso_code,year,death_rate,forest_area_pct,gdp_per_capita,health_expenditure_pc,life_expectancy,physicians_per_1000,pm25_air_pollution,population_total,...,infant_mortality_who,life_expectancy_who,maternal_mortality_who,country_name,is_blue_zone,blue_zone_region,latitude,longitude,effective_gravity,gravity_deviation
0,AFG,1960,31.672,NaN,NaN,NaN,32.799,0.035,NaN,9035043.0,...,NaN,NaN,NaN,Afghanistan,0,NaN,33.9391,67.71,9.796446,-0.010204
1,AFG,1961,31.129,NaN,NaN,NaN,33.291,NaN,NaN,9214083.0,...,NaN,NaN,NaN,Afghanistan,0,NaN,33.9391,67.71,9.796446,-0.010204
2,AFG,1962,30.630,NaN,NaN,NaN,33.757,NaN,NaN,9404406.0,...,NaN,NaN,NaN,Afghanistan,0,NaN,33.9391,67.71,9.796446,-0.010204
3,AFG,1963,30.168,NaN,NaN,NaN,34.201,NaN,NaN,9604487.0,...,NaN,NaN,NaN,Afghanistan,0,NaN,33.9391,67.71,9.796446,-0.010204
4,AFG,1964,29.672,NaN,NaN,NaN,34.673,NaN,NaN,9814318.0,...,NaN,NaN,NaN,Afghanistan,0,NaN,33.9391,67.71,9.796446,-0.010204


In [3]:
df.dtypes

iso_code                   object
year                        int64
death_rate                float64
forest_area_pct           float64
gdp_per_capita            float64
health_expenditure_pc     float64
life_expectancy           float64
physicians_per_1000       float64
pm25_air_pollution        float64
population_total          float64
urban_population_pct      float64
infant_mortality_who      float64
life_expectancy_who       float64
maternal_mortality_who    float64
country_name               object
is_blue_zone                int64
blue_zone_region           object
latitude                  float64
longitude                 float64
effective_gravity         float64
gravity_deviation         float64
dtype: object

## 2. Data Completeness Analysis

In [4]:
# Completeness by indicator
indicators = ['life_expectancy', 'gdp_per_capita', 'physicians_per_1000',
              'urban_population_pct', 'pm25_air_pollution', 'health_expenditure_pc',
              'death_rate', 'forest_area_pct', 'population_total',
              'infant_mortality_who', 'life_expectancy_who', 'maternal_mortality_who']

completeness = []
for col in indicators:
    if col in df.columns:
        n_valid = df[col].notna().sum()
        pct = n_valid / len(df) * 100
        earliest = df[df[col].notna()]['year'].min() if n_valid > 0 else None
        latest = df[df[col].notna()]['year'].max() if n_valid > 0 else None
        completeness.append({
            'indicator': col,
            'n_valid': n_valid,
            'pct_filled': round(pct, 1),
            'earliest_year': earliest,
            'latest_year': latest
        })

comp_df = pd.DataFrame(completeness).sort_values('pct_filled', ascending=False)
comp_df

,indicator,n_valid,pct_filled,earliest_year,latest_year
3,urban_population_pct,5952,100.0,1960,2023
6,death_rate,5952,100.0,1960,2023
8,population_total,5952,100.0,1960,2023
0,life_expectancy,5948,99.9,1960,2023
1,gdp_per_capita,5306,89.1,1960,2023
2,physicians_per_1000,3191,53.6,1960,2023
7,forest_area_pct,3085,51.8,1990,2023
4,pm25_air_pollution,2883,48.4,1990,2020
5,health_expenditure_pc,2188,36.8,2000,2023
9,infant_mortality_who,2046,34.4,2000,2021


In [5]:
# Visual: completeness heatmap by country and indicator
bz_isos = ['USA', 'JPN', 'ITA', 'GRC', 'CRI']
sample_countries = bz_isos + ['CHN', 'DEU', 'BRA', 'NGA', 'AUS', 'KOR', 'ZAF', 'MEX', 'IND', 'GBR']

key_indicators = ['life_expectancy', 'gdp_per_capita', 'physicians_per_1000',
                  'health_expenditure_pc', 'pm25_air_pollution', 'death_rate']

heatmap_data = []
for iso in sample_countries:
    country = df[df['iso_code'] == iso]
    row = {'country': iso}
    for ind in key_indicators:
        if ind in country.columns:
            row[ind] = round(country[ind].notna().mean() * 100, 0)
        else:
            row[ind] = 0
    heatmap_data.append(row)

hm = pd.DataFrame(heatmap_data).set_index('country')

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(hm, annot=True, fmt='.0f', cmap='RdYlGn', vmin=0, vmax=100,
            cbar_kws={'label': '% of years with data'}, ax=ax)
ax.set_title('Data Completeness by Country and Indicator (% of years 1960-2023)')
ax.set_ylabel('')

# Mark Blue Zone countries
for i, label in enumerate(ax.get_yticklabels()):
    if label.get_text() in bz_isos:
        label.set_fontweight('bold')
        label.set_color('darkblue')

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, 'outputs', 'figures', 'data_completeness_heatmap.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved: data_completeness_heatmap.png')

Saved: data_completeness_heatmap.png


/tmp/ipykernel_1567546/2525080390.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Blue Zone Country Overview

In [6]:
bz_names = {'USA': 'United States (Loma Linda)', 'JPN': 'Japan (Okinawa)',
            'ITA': 'Italy (Sardinia)', 'GRC': 'Greece (Ikaria)', 'CRI': 'Costa Rica (Nicoya)'}

for iso in bz_isos:
    c = df[df['iso_code'] == iso]
    le = c['life_expectancy'].dropna()
    print(f'{bz_names[iso]}:')
    print(f'  Years with LE data: {len(le)} ({c["year"].min()}-{c["year"].max()})')
    if len(le) > 0:
        print(f'  LE range: {le.min():.1f} - {le.max():.1f} years')
        print(f'  Total improvement: {le.max() - le.min():.1f} years')
    gdp = c['gdp_per_capita'].dropna()
    if len(gdp) > 0:
        print(f'  GDP per capita range: ${gdp.min():,.0f} - ${gdp.max():,.0f}')
    print()

United States (Loma Linda):
  Years with LE data: 64 (1960-2023)
  LE range: 69.8 - 78.8 years
  Total improvement: 9.1 years
  GDP per capita range: $3,000 - $81,032

Japan (Okinawa):
  Years with LE data: 64 (1960-2023)
  LE range: 67.7 - 84.6 years
  Total improvement: 16.9 years
  GDP per capita range: $509 - $49,145

Italy (Sardinia):
  Years with LE data: 64 (1960-2023)
  LE range: 69.1 - 83.7 years
  Total improvement: 14.6 years
  GDP per capita range: $837 - $40,829

Greece (Ikaria):
  Years with LE data: 64 (1960-2023)
  LE range: 70.2 - 81.8 years
  Total improvement: 11.6 years
  GDP per capita range: $513 - $31,696

Costa Rica (Nicoya):
  Years with LE data: 64 (1960-2023)
  LE range: 63.5 - 80.8 years
  Total improvement: 17.3 years
  GDP per capita range: $335 - $16,942



## 4. Life Expectancy Distribution Over Time

In [7]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

key_years = [1960, 1970, 1980, 1990, 2000, 2020]

for i, year in enumerate(key_years):
    ax = axes[i]
    year_data = df[(df['year'] == year) & (df['life_expectancy'].notna())]
    
    non_bz = year_data[year_data['is_blue_zone'] == 0]['life_expectancy']
    bz = year_data[year_data['is_blue_zone'] == 1]['life_expectancy']
    
    if len(non_bz) > 0:
        ax.hist(non_bz, bins=20, alpha=0.6, color='steelblue', label='Other countries')
    if len(bz) > 0:
        for val in bz:
            ax.axvline(val, color='red', linestyle='--', alpha=0.7)
        ax.axvline(bz.mean(), color='red', linewidth=2, label=f'Blue Zone avg: {bz.mean():.1f}')
    
    ax.set_title(f'{year} (n={len(year_data)})')
    ax.set_xlabel('Life Expectancy (years)')
    ax.set_xlim(25, 90)
    ax.legend(fontsize=8)

fig.suptitle('Global Life Expectancy Distribution with Blue Zone Countries Marked', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, 'outputs', 'figures', 'le_distribution_over_time.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved: le_distribution_over_time.png')

Saved: le_distribution_over_time.png


/tmp/ipykernel_1567546/3431147501.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Correlation Matrix of Key Indicators

In [8]:
# Use most recent year with good coverage for correlations
recent = df[df['year'] >= 2015].groupby('iso_code').last().reset_index()

corr_cols = ['life_expectancy', 'gdp_per_capita', 'physicians_per_1000',
             'health_expenditure_pc', 'urban_population_pct', 'pm25_air_pollution',
             'death_rate']
corr_cols = [c for c in corr_cols if c in recent.columns]

corr_matrix = recent[corr_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax)
ax.set_title('Correlation Matrix of Health and Development Indicators (2015+)')
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, 'outputs', 'figures', 'correlation_matrix.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved: correlation_matrix.png')

Saved: correlation_matrix.png


/tmp/ipykernel_1567546/2153535317.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Summary Statistics

In [9]:
# Summary stats for numerical columns
numeric_cols = ['life_expectancy', 'gdp_per_capita', 'physicians_per_1000',
                'health_expenditure_pc', 'urban_population_pct', 'death_rate']
numeric_cols = [c for c in numeric_cols if c in df.columns]

summary = df[numeric_cols].describe().round(2)
summary

,life_expectancy,gdp_per_capita,physicians_per_1000,health_expenditure_pc,urban_population_pct,death_rate
count,5948.00,5306.00,3191.00,2188.00,5952.00,5952.00
mean,67.78,10038.42,1.97,1509.48,56.00,9.82
std,10.14,16622.24,1.35,2126.99,23.30,4.09
min,26.52,11.80,0.01,4.47,3.00,2.17
25%,61.61,793.91,0.86,118.71,37.57,7.04
50%,70.36,2844.92,1.84,475.16,59.11,9.20
75%,75.06,11141.60,3.03,2046.36,75.05,11.50
max,84.56,134965.82,9.34,13473.19,100.00,39.88


In [10]:
# Blue Zone vs Non-Blue Zone comparison (most recent data)
recent_all = df[df['year'] >= 2018].copy()
comparison = recent_all.groupby('is_blue_zone')[numeric_cols].mean().round(2)
comparison.index = ['Non-Blue Zone', 'Blue Zone']
comparison

,life_expectancy,gdp_per_capita,physicians_per_1000,health_expenditure_pc,urban_population_pct,death_rate
Non-Blue Zone,74.60,21074.98,2.51,1810.13,66.26,8.44
Blue Zone,81.19,35423.21,3.85,4389.80,79.51,10.16


In [11]:
print('\nNotebook 01 complete. Key findings:')
print(f'- {df["iso_code"].nunique()} countries across {df["year"].max() - df["year"].min() + 1} years')
print(f'- {len(df):,} total observations')
print(f'- Life expectancy coverage: {df["life_expectancy"].notna().mean()*100:.1f}%')
print(f'- All 5 Blue Zone countries have complete LE data')


Notebook 01 complete. Key findings:
- 93 countries across 64 years
- 5,952 total observations
- Life expectancy coverage: 99.9%
- All 5 Blue Zone countries have complete LE data
